# QuickTalk: train a small model from scratch (Colab, A100)

Trains a GPT-style model on `train/build/` from the `sraivante/quicktalk` repo: first on plain text (books + behaviour
passages), then on chats (behaviour Q&A + 12 conversation patterns).

**Before you start**
1. *Runtime > Change runtime type > A100 GPU*. The notebook stops if the GPU is not an A100.
2. The repo is public, so no token is needed. (Only if it is made private later: add a Colab secret named
   `GITHUB_TOKEN` with a GitHub token that can read the repo.)
3. *Runtime > Run all*.

**Crashes, disconnects and idle timeouts**
- Everything is saved to Google Drive (`MyDrive/quicktalk_run`): tokenizer, token files, checkpoints every 5 minutes,
  logs. If the runtime is killed, reconnect and *Run all* again: finished steps are skipped and training resumes from
  the last checkpoint.
- Training runs in the background, so closing the browser tab does not stop it while the runtime is alive. With Colab
  Pro+ you can also turn on background execution so the runtime keeps running after you close the tab.
- The whole run is short (see the plan cell for the step count), so a session timeout is unlikely to hit it.

**Model size:** the plan cell uses your rule of 10 training tokens per parameter (not Chinchilla's 20).

**Runs:** each run lives in its own Drive folder (`RUN` in cell 2), so earlier models are kept.
- Run 1 `quicktalk_run`: 4 + 2 passes, 8.9M parameters.
- Run 2 `quicktalk_run2`: 6 + 3 passes, 12.6M parameters. Best chat eval loss so far (3.31).
- Run 3 `quicktalk_run3`: run 2 + conversation patterns repeated 8x over 3 passes (24 views each): overfit (train
  loss 2.1, eval loss 3.81).
- **Run 4 `quicktalk_run4` (this version):** run 2's pretrained weights + patterns repeated 3x over 2 passes (6 views
  each). Only the chat stage trains (~2 minutes).
- Cell 9 now asks every question in a fresh conversation (the old test carried earlier answers into later ones) and
  uses a repetition penalty, so all runs are compared fairly.

In [ ]:
# 1. Check the GPU: A100 only
import subprocess, torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > A100 GPU'
name = torch.cuda.get_device_name(0)
print('GPU:', name)
assert 'A100' in name, f'This runtime has {name}, not an A100. Change the runtime type to A100 and run again.'

In [ ]:
# 2. Google Drive holds all state, so nothing is lost if the runtime restarts
import os, shutil
RUN = 'quicktalk_run4'          # this experiment's folder; earlier runs stay untouched
FROM = 'quicktalk_run2'         # reuse this run's tokenizer, pretraining tokens, model size and pretrained weights
CHAT_REPEAT, SFT_EPOCHS = 3, 2  # repeat the 12 conversation-pattern types 3x; 2 passes over the chats
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
    except Exception as e:
        print('Drive mount failed:', e)
        print("""Fix, then run this cell again:
 1. In the Google sign-in pop-up, choose the SAME account you use for Colab, and on the permissions page tick
    'Select all' (full Drive access). Unticking any box makes the mount fail.
 2. Allow pop-ups and third-party cookies for colab.research.google.com.
 3. Or click the folder icon on the left, then the 'Mount Drive' icon, and run this cell again.
 4. Still failing: Runtime > Disconnect and delete runtime, reconnect (A100), Run all.""")
        raise
BASE = '/content/drive/MyDrive'
WORK, SRCRUN = f'{BASE}/{RUN}', f'{BASE}/{FROM}'
os.makedirs(f'{WORK}/data', exist_ok=True); os.makedirs(f'{WORK}/checkpoints', exist_ok=True)
for f in ['tokenizer.json', 'plan.json', 'data/pretrain_train.bin', 'data/pretrain_val.bin', 'checkpoints/pretrain_final.pt']:
    if not os.path.exists(f'{WORK}/{f}'):
        assert os.path.exists(f'{SRCRUN}/{f}'), f'{SRCRUN}/{f} missing: run {FROM} first'
        shutil.copy(f'{SRCRUN}/{f}', f'{WORK}/{f}'); print('copied', f, 'from', FROM)
print('work dir:', WORK, sorted(os.listdir(WORK)))

In [ ]:
# 3. Get the code and the built data (sparse clone: only train/build and train/colab)
import os, subprocess
REPO, BRANCH, SRC = 'sraivante/quicktalk', 'claude/intelligent-ride-oxqjfy', '/content/quicktalk'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.exists(f'{SRC}/train/build/manifest.json'):
    subprocess.run(['rm', '-rf', SRC])
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, '--filter=blob:none', '--sparse', url, SRC], check=True)
    subprocess.run(['git', '-C', SRC, 'sparse-checkout', 'set', 'train/build', 'train/colab'], check=True)
print(sorted(os.listdir(f'{SRC}/train/build')))
LM = f'{SRC}/train/colab/quicktalk_lm.py'
DATA = f'{SRC}/train/build'

In [ ]:
# 4. Libraries
!pip -q install tokenizers

In [ ]:
# 5. Chat token files with the conversation patterns repeated (skips itself if done), then update the plan
!python {LM} prepare --data {DATA} --workdir {WORK} --chat-repeat {CHAT_REPEAT}
import json, os
plan = json.load(open(f'{WORK}/plan.json'))          # same model size as the source run (needed to reuse its weights)
plan['sft_tokens'] = os.path.getsize(f'{WORK}/data/sft_train.bin') // 2
plan['sft_epochs'] = SFT_EPOCHS; plan['chat_repeat'] = CHAT_REPEAT
json.dump(plan, open(f'{WORK}/plan.json', 'w'), indent=1)
print({k: plan[k] for k in ('params', 'd', 'layers', 'sft_tokens', 'sft_epochs', 'chat_repeat')})

In [ ]:
# 6. Start training in the background (pretrain, then chat fine-tuning). Safe to re-run: it resumes.
import os, subprocess
os.makedirs('/content/cache', exist_ok=True)
subprocess.run(f'rsync -a {WORK}/data/ /content/cache/data/', shell=True, check=True)   # local copy reads faster than Drive
LOG = f'{WORK}/train.log'
cmd = (f'python {LM} train --workdir {WORK} --data-dir /content/cache/data --stage pretrain && '
       f'python {LM} train --workdir {WORK} --data-dir /content/cache/data --stage sft')
running = subprocess.run("pgrep -f '[q]uicktalk_lm.py train'", shell=True, capture_output=True).stdout.strip()
if running:
    print('training is already running; watch it in the next cell')
else:
    subprocess.Popen(f'nohup bash -c "{cmd}" >> {LOG} 2>&1 &', shell=True)
    print('started; log:', LOG)

In [ ]:
# 7. Watch progress (re-run this cell any time; stop it with the stop button, training keeps going)
import time, subprocess, os
LOG = f'{WORK}/train.log'
while True:
    out = subprocess.run(f'tail -n 6 {LOG}', shell=True, capture_output=True, text=True).stdout
    print(out, flush=True)
    done = os.path.exists(f'{WORK}/checkpoints/sft_final.pt')
    alive = subprocess.run("pgrep -f '[q]uicktalk_lm.py train'", shell=True, capture_output=True).stdout.strip()
    if done: print('DONE: model at', f'{WORK}/checkpoints/sft_final.pt'); break
    if not alive: print('training is not running: check the log above, then re-run cell 6 to resume'); break
    time.sleep(60)

In [ ]:
# 8. Loss curves (step, train loss, eval loss)
import pandas as pd, matplotlib.pyplot as plt, os
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for i, st in enumerate(['pretrain', 'sft']):
    p = f'{WORK}/log_{st}.csv'
    if os.path.exists(p):
        d = pd.read_csv(p, names=['step', 'train_loss', 'eval_loss'])
        ax[i].plot(d['step'], d['train_loss'], label='train')
        ax[i].plot(d['step'], d['eval_loss'], label='eval')
        ax[i].set_title(st); ax[i].legend()
plt.show()

In [ ]:
# 9. Talk to the model: every run gets the same questions, each in a fresh conversation
import os
PROMPTS = ' '.join(f'--prompt "{q}"' for q in [
    'Hi! How was your day?',
    'What should I do if my friend is upset with me?',
    'Correct this: She dont like apples.',
    'What does the word diligent mean?',
    'Kal mera exam hai, I am very nervous. What should I do?',
    'Write a short note to thank my teacher.',
])
for run in sorted(os.listdir(BASE)):
    if run.startswith('quicktalk_run') and os.path.exists(f'{BASE}/{run}/checkpoints/sft_final.pt'):
        print(f'\n===== {run} =====')
        !python {LM} chat --workdir {BASE}/{run} {PROMPTS} --temperature 0.6

In [ ]:
# 10. Compare runs: model size and final eval loss (lower is better)
import json, os, pandas as pd
rows = []
for run in sorted(os.listdir(BASE)):
    w = f'{BASE}/{run}'
    if run.startswith('quicktalk_run') and os.path.exists(f'{w}/plan.json'):
        pl = json.load(open(f'{w}/plan.json')); r = {'run': run, 'params_M': round(pl['params'] / 1e6, 1),
             'passes': f"{pl['pretrain_epochs']}+{pl['sft_epochs']}", 'chat_repeat': pl.get('chat_repeat', 1)}
        for st in ('pretrain', 'sft'):
            p = f'{w}/log_{st}.csv'
            if os.path.exists(p): r[f'{st}_eval_loss'] = pd.read_csv(p, names=['s', 't', 'e'])['e'].iloc[-1]
        rows.append(r)
pd.DataFrame(rows)